# LTspice 인터랙티브 회로 실험실
슬라이더를 움직이면 LTspice가 즉시 다시 시뮬레이션하고 그래프가 갱신됩니다.

먼저 아래 셀을 실행하세요.

In [1]:
import numpy as np
from circuitlab import dashboard
import schematics

## 1. RC 저역통과 필터 - 주파수 응답 (보드 선도)
**관찰 포인트**: R 또는 C를 키우면 차단주파수 fc가 왼쪽(낮은 주파수)으로 이동합니다. fc에서 크기는 -3 dB, 위상은 -45°.

In [6]:
dashboard(
    "circuits/rc_lowpass_ac.cir",
    sliders={"R": (10, 1e6, 1e3, "Ω"), "C": (1e-9, 1e-3, 1e-6, "F")},
    views=[
        dict(title="크기 (Gain)", traces=["V(out)"], transform="mag_db", xlog=True, xlabel="주파수 (Hz)", ylabel="dB"),
        dict(title="위상 (Phase)", traces=["V(out)"], transform="phase_deg", xlog=True, xlabel="주파수 (Hz)", ylabel="도(°)"),
    ],
    info=lambda p: f"차단주파수 fc = 1/(2πRC) = <b>{1/(2*np.pi*p['R']*p['C']):,.1f} Hz</b>",
    title="RC 저역통과 필터 (AC)",
    schematic=schematics.rc_lowpass,
)

## 2. RC 회로 - 시간 응답 (충전/방전)
**관찰 포인트**: 시정수 τ = RC. 커패시터는 τ 후 약 63%까지 충전됩니다. (시간축은 R·C에 맞춰 자동 조정)

In [3]:
dashboard(
    "circuits/rc_lowpass_tran.cir",
    sliders={"R": (100, 1e6, 1e3, "Ω"), "C": (1e-9, 1e-3, 1e-6, "F")},
    views=[dict(title="입력 vs 출력", traces=["V(in)", "V(out)"], xlabel="시간 (s)", ylabel="V")],
    info=lambda p: f"시정수 τ = RC = <b>{p['R']*p['C']*1e3:.4g} ms</b>",
    title="RC 충방전 (Transient)",
    schematic=schematics.rc_lowpass,
)

## 3. 직렬 RLC 공진 회로
**관찰 포인트**: 공진주파수에서 출력이 최대. R을 줄이면 피크가 뾰족해집니다(Q값 증가).

In [4]:
dashboard(
    "circuits/rlc_series_ac.cir",
    sliders={"R": (1, 1e4, 100, "Ω"), "L": (1e-6, 1, 1e-3, "H"), "C": (1e-9, 1e-3, 1e-6, "F")},
    views=[dict(title="크기 (Gain)", traces=["V(out)"], transform="mag_db", xlog=True, xlabel="주파수 (Hz)", ylabel="dB")],
    info=lambda p: f"공진주파수 f0 = 1/(2π√LC) = <b>{1/(2*np.pi*np.sqrt(p['L']*p['C'])):,.1f} Hz</b>, Q = {np.sqrt(p['L']/p['C'])/p['R']:.2f}",
    title="직렬 RLC (AC)",
    schematic=schematics.rlc_series,
)

## 4. 내가 만든 넷리스트 점검
`.cir` 파일을 수정한 뒤 이 셀만 다시 실행하면 ngspice 로그(문법/연결 오류)와 결과 그래프를 바로 확인합니다.
`$PARAMS$` 자리가 있는 파일은 `check("파일", R=1e3, C=1e-6)`처럼 값을 넘기세요.

In [ ]:
from circuitlab import check
check("circuits/ch1/lab02.cir")

## 5. 9V 건전지 + 다이오드 + 저항 (직류)
**관찰 포인트**: 전압을 올리면 전류가 급격히(지수적으로) 늘고, 저항을 키우면 전류가 거의 R에 반비례해 줄어듭니다. 다이오드 양단 전압 V_D는 전류가 변해도 1.5V 근처에서 크게 변하지 않습니다.

In [ ]:
from circuitlab import op_explorer

op_explorer(
    "circuits/ch1/lab02_dc.cir",
    sliders={"V": (0.1, 12, 9, "V"), "R": (10, 500e3, 1.5e3, "Ω")},
    outputs={
        "I_D": (lambda r: -r["i(v1)"], "A"),              # 전원이 내보내는 전류 = 다이오드 전류
        "V_D": (lambda r: r["v(in)"] - r["v(out)"], "V"),  # 다이오드 양단 전압
        "V_R": (lambda r: r["v(out)"], "V"),               # 저항 양단 전압
    },
    curves=[
        dict(x="R", y="I_D", lo=10, hi=500e3, log=True, ylog=True),  # 저항 10Ω~500kΩ 스윕
        dict(x="V", y="I_D", lo=0.1, hi=12),                         # 전압 스윕
    ],
    title="건전지 - 다이오드 - 저항 (DC)",
)